# Explore beam transport with ImpactX

A **100 MeV electron bunch** enters HTU: quadrupole magnets focus it, a chicane bends its path, and more magnets guide it toward an undulator. Follow how the beam shape changes along the beamline.

We track a **25 pC Gaussian bunch** with the upstream ImpactX HTU lattice. This source is independent of the WarpX run. Each macroparticle represents many electrons; increasing their number samples the same physical bunch more finely.

**Use the WarpX CPU kernel (which also includes ImpactX)** and run from this notebook's `htu/beamline_impactx` folder. Start with the three settings below. The first cell downloads the [upstream HTU lattice](https://github.com/BLAST-ImpactX/impactx/blob/26.09/examples/htu_beamline/htu_lattice.py) from ImpactX 26.09 only if `htu_lattice.py` is missing. Existing local copies are kept. The ImpactX simulation runs directly in this notebook. The [helper script](impactx_helpers.py) contains only analysis and plotting utilities.

🔬 **Model boundaries:** prescribed magnets, no space charge, no radiation, and no modeled apertures. The screens record the simulated beam without clipping it. With no collective fields, particles track independently—a useful contrast to WarpX's PIC workload.


In [ ]:
import json
import sys
import tempfile
import time
from contextlib import chdir
from pathlib import Path
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import openpmd_api as io

# Local modules: import after the path setup and download above.
from htu_lattice import get_lattice
from impactx import ImpactX, distribution, elements, twiss
from impactx_helpers import beam_explorer
from scipy.constants import c, e, m_e

# Locate this notebook's folder even if the kernel started in htu/.
notebook_dir = Path.cwd()
if not (notebook_dir / "impactx_helpers.py").is_file():
    notebook_dir = notebook_dir / "beamline_impactx"
sys.path.insert(0, str(notebook_dir))

# Download the upstream lattice once; keep any existing local copy.
lattice_file = notebook_dir / "htu_lattice.py"
if not lattice_file.exists():
    url = "https://raw.githubusercontent.com/BLAST-ImpactX/impactx/26.09/examples/htu_beamline/htu_lattice.py"
    with urlopen(url, timeout=30) as response:
        lattice_file.write_bytes(response.read())

In [ ]:
particles = 10_000
total_energies_MeV = [100]  # Total energy, including rest energy.
chicane_r56_um = 200.0  # Upstream chicane control setting.

## 1 · Build and run the ImpactX simulation

The function below contains the complete simulation and runs directly in this kernel. Change the settings above to explore different particle counts or beam energies.

Read the five numbered sections:

1. `ImpactX()` creates the simulation; diagnostics record its evolution.
2. The reference particle defines the charge, mass, and reference energy. The API takes **kinetic** energy, so we subtract rest energy from total energy.
3. `twiss(...)` supplies Gaussian distribution parameters. Dividing the 1.5 µm normalized transverse emittance by βγ gives geometric emittance. `sim.add_particles(...)` samples the bunch. `mean_pt` shifts its energy while retaining the 100 MeV reference.
4. `sim.lattice.extend(...)` sets the ordered magnets and diagnostic screens.
5. `sim.track_particles()` transports the bunch; `sim.finalize()` closes the simulation.

**Predict:** how will the first focusing magnets change the beam shape?

Rerun this definition cell after editing it, then run the call below.


In [ ]:
def run_beams(particles=10_000, energies=(100,), chicane_r56_um=200.0, cpu_threads=2):
    """Run ImpactX in this kernel and save each case in a fresh folder."""
    mass_MeV = m_e * c**2 / e / 1e6
    if particles < 2:
        raise ValueError("Use at least two particles.")
    if any(not np.isfinite(energy) or energy <= mass_MeV for energy in energies):
        raise ValueError(
            "Total energies must be finite and exceed the electron rest energy."
        )
    if not np.isfinite(chicane_r56_um) or chicane_r56_um < 0:
        raise ValueError("The chicane setting must be finite and nonnegative.")
    runs = notebook_dir.parent / "runs"
    runs.mkdir(exist_ok=True)
    run = Path(tempfile.mkdtemp(prefix="transport-", dir=runs))
    performance = []

    for energy in energies:
        case = run / f"{energy:g}MeV"
        case.mkdir()
        started = time.perf_counter()
        # ImpactX writes diagnostics relative to its working directory.
        # chdir restores the notebook directory even if the simulation fails.
        with chdir(case):
            sim = ImpactX()
            try:
                # 1. Configure independent-particle tracking and diagnostics.
                sim.omp_threads = cpu_threads  # OpenMP threads, read by init_grids()
                sim.particle_shape = 2
                sim.space_charge = False
                sim.slice_step_diagnostics = True
                sim.init_grids()

                # 2. Set the electron reference particle (kinetic energy in MeV).
                reference_total_energy_MeV = 100.0
                ref = sim.beam.ref
                ref.set_charge_qe(-1.0).set_mass_MeV(mass_MeV).set_kin_energy_MeV(
                    reference_total_energy_MeV - mass_MeV
                )

                # 3. Sample a 25 pC Gaussian bunch using Twiss parameters.
                bg = np.sqrt((reference_total_energy_MeV / mass_MeV) ** 2 - 1)
                sigma_t, sigma_pt = 1e-6, 0.025
                mean_pt = -(energy - reference_total_energy_MeV) / mass_MeV / bg
                bunch = distribution.Gaussian(
                    **twiss(
                        beta_x=0.002,
                        beta_y=0.002,
                        beta_t=sigma_t / sigma_pt,
                        emitt_x=1.5e-6 / bg,
                        emitt_y=1.5e-6 / bg,
                        emitt_t=sigma_t * sigma_pt,
                        mean_pt=mean_pt,
                    )
                )
                sim.add_particles(bunch_charge=25e-12, distr=bunch, npart=particles)

                # 4. Load the magnets and add entrance/exit snapshots.
                monitor = elements.BeamMonitor("monitor", backend="h5")
                sim.lattice.extend(
                    [
                        monitor,
                        *get_lattice("impactx", chicane_r56=chicane_r56_um),
                        monitor,
                    ]
                )

                # 5. Track through the lattice and close diagnostic files.
                sim.track_particles()
            finally:
                sim.finalize()

        elapsed = time.perf_counter() - started
        size = sum(p.stat().st_size for p in case.rglob("*") if p.is_file()) / 2**20
        performance.append(
            dict(
                particles=particles,
                total_energy_MeV=energy,
                chicane_r56_um=chicane_r56_um,
                requested_cpu_threads=cpu_threads,
                elapsed_seconds=elapsed,
                output_MiB=size,
            )
        )
        print(
            f"{energy:g} MeV | {particles:,} particles | {elapsed:.2f} s | {size:.1f} MiB"
        )

    (run / "performance.json").write_text(json.dumps(performance, indent=2) + "\n")
    return run

### Run and measure

Call the function defined above. Each call creates a fresh folder under `../runs/`, with diagnostics and `performance.json`. ImpactX prints its progress in the notebook. The timer covers simulation setup, particle generation, tracking, and diagnostic output; it excludes kernel startup and plotting.

`run_beams` requests two CPU threads via `sim.omp_threads`; pass, e.g., `cpu_threads=4` to change that.


In [ ]:
run = run_beams(particles, total_energies_MeV, chicane_r56_um)

## 2 · Open a diagnostic and plot the beam

Before using the interactive viewer, open one screen directly with [openPMD-api](https://openpmd-api.readthedocs.io/en/0.17.1/analysis/pandas.html). Select the first beam energy and `TCPhosphor`, the screen after the initial focusing magnets.

An openPMD series contains iterations; each iteration contains particle species. `to_df()` loads the `beam` species into a pandas table, with one row per macroparticle. Inspect its coordinate and weight columns below. Close the series after loading the table.


In [ ]:
energy = total_energies_MeV[0]
screen_name = "TCPhosphor"
case = run / f"{energy:g}MeV"
screen_file = case / "diags/openPMD" / f"{screen_name}.h5"
series = io.Series(str(screen_file), io.Access.read_only)
try:
    iteration = min(series.iterations)
    beam = series.iterations[iteration].particles["beam"].to_df()
finally:
    series.close()
beam[["position_x", "position_y", "weighting"]].head()

Extract the transverse positions in meters and macroparticle weights from the table. Convert positions to millimeters and weights to charge magnitude in pC, then plot the beam with Matplotlib. The color shows **charge per bin**, not charge density. Try another screen name after this first example.


In [ ]:
x = beam["position_x"].to_numpy()
y = beam["position_y"].to_numpy()
w = beam["weighting"].to_numpy()

fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
hist = ax.hist2d(x * 1e3, y * 1e3, bins=50, weights=w * e * 1e12, cmap="magma")
fig.colorbar(hist[3], ax=ax, label="Charge per bin (pC)")
ax.set(xlabel="x (mm)", ylabel="y (mm)", title=f"{screen_name} · {energy:g} MeV")
ax.set_aspect("equal")
plt.show()

## 3 · Explore the beam screens 🔍

Press **Play screens** or drag the slider. These are saved transverse snapshots, not a movie in physical time. Turn off **Zoom to fit** to compare sizes on fixed axes and see how much the beam expands from the source.

Find the first focusing screen (`TCPhosphor`), the middle of the chicane (`ChicaneSlit`), and the exit. Does the beam stay round? Read the axes: zoomed maps can switch between µm and mm. Colors show charge per bin, not density per unit area.

The viewer also saves `beam_explorer.html` (open it in a browser if notebook HTML is blocked) in the run folder.


In [ ]:
beam_explorer(run)